In [1]:
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import os
import matplotlib.pyplot as plt

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

In [3]:
data = '/content/Blink Detection Dataset.zip'
img_h = 32
img_w = 32
batch = 32

if data.endswith('.zip'):
    import zipfile
    import os
    extracted_dir = data.replace('.zip', '')
    if not os.path.exists(extracted_dir):
        print(f"Extracting {data} to {extracted_dir}...")
        with zipfile.ZipFile(data, 'r') as zip_ref:
            zip_ref.extractall(os.path.dirname(extracted_dir))
        print("Extraction complete.")
    data = extracted_dir

def build_and_train_classifier():
    print(f"Loading dataset from: {data}")
    train_ds = tf.keras.utils.image_dataset_from_directory(
        data,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=(img_h, img_w),
        batch_size=batch
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        data,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=(img_h, img_w),
        batch_size=batch
    )

    print("Building CNN Classifier...")
    model = models.Sequential([
        layers.Rescaling(1./255, input_shape=(img_h, img_w, 3)),
        layers.Conv2D(16, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid') # Binary output: 0 (Close) or 1 (Open)
    ])

    model.compile(optimizer='adam',
                  loss=tf.keras.losses.BinaryCrossentropy(),
                  metrics=['accuracy'])

    print("Training model...")

    history = model.fit(train_ds, validation_data=val_ds, epochs=10)

    print("Training complete!")
    return model

blink_model = build_and_train_classifier()

Extracting /content/Blink Detection Dataset.zip to /content/Blink Detection Dataset...
Extraction complete.
Loading dataset from: /content/Blink Detection Dataset
Found 41649 files belonging to 2 classes.
Using 33320 files for training.
Found 41649 files belonging to 2 classes.
Using 8329 files for validation.
Building CNN Classifier...
Training model...
Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1042/1042 ━━━━━━━━━━━━━━━━━━━━ 38s 34ms/step - accuracy: 0.9447 - loss: 0.1410 - val_accuracy: 0.9586 - val_loss: 0.1161
Epoch 2/10
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 34s 32ms/step - accuracy: 0.9778 - loss: 0.0655 - val_accuracy: 0.9783 - val_loss: 0.0588
Epoch 3/10
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 37s 35ms/step - accuracy: 0.9859 - loss: 0.0433 - val_accuracy: 0.9830 - val_loss: 0.0465
Epoch 4/10
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 35s 33ms/step - accuracy: 0.9888 - loss: 0.0325 - val_accuracy: 0.9845 - val_loss: 0.0471
Epoch 5/10
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 41s 33ms/step - accuracy: 0.9903 - loss: 0.0276 - val_accuracy: 0.9886 - val_loss: 0.0350
Epoch 6/10
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 32s 31ms/step - accuracy: 0.9926 - loss: 0.0217 - val_accuracy: 0.9882 - val_loss: 0.0373
Epoch 7/10
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 32s 31ms/step - accuracy: 0.9938 - loss: 0.0178 - val_accuracy: 0.9918 - val_loss: 0.0261
Epoch 8/10
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 45s 35ms/step - accuracy: 0.9946 - loss: 0.01

In [4]:
LEFT_EYE_INDICES = [33, 160, 158, 133, 153, 144]
RIGHT_EYE_INDICES = [362, 385, 387, 263, 373, 380]

def get_eye_crop(frame, face_landmarks, eye_indices, padding=5):
    ih, iw, _ = frame.shape
    x_coords = []
    y_coords = []

    for idx in eye_indices:
        landmark = face_landmarks.landmark[idx]
        x_coords.append(int(landmark.x * iw))
        y_coords.append(int(landmark.y * ih))

    # Get bounding box
    x_min, x_max = min(x_coords), max(x_coords)
    y_min, y_max = min(y_coords), max(y_coords)

    # Add padding to ensure the whole eye is captured
    x_min = max(0, x_min - padding)
    y_min = max(0, y_min - padding)
    x_max = min(iw, x_max + padding)
    y_max = min(ih, y_max + padding)

    eye_crop = frame[y_min:y_max, x_min:x_max]
    return eye_crop

def predict_eye_state(eye_img, model):
    if eye_img.size == 0:
        return "Open"

    img_resized = cv2.resize(eye_img, (img_w, img_h))
    img_array = tf.expand_dims(img_resized, 0)

    prediction = model.predict(img_array, verbose=0)[0][0]
    return "Open" if prediction > 0.5 else "Close"

In [7]:
Video = '/content/patient.mp4'

cap = cv2.VideoCapture(Video)

left_eye_state = "Open"
right_eye_state = "Open"

left_blink_count = 0
right_blink_count = 0

print("Processing video")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # MediaPipe requires RGB images
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_frame)

    if results.multi_face_landmarks:
        for face_landmarks in results.multi_face_landmarks:

            # Process Left Eye
            left_eye_img = get_eye_crop(frame, face_landmarks, LEFT_EYE_INDICES)
            current_left_state = predict_eye_state(left_eye_img, blink_model)

            if left_eye_state == "Close" and current_left_state == "Open":
                left_blink_count += 1
            left_eye_state = current_left_state

            # Process Right
            right_eye_img = get_eye_crop(frame, face_landmarks, RIGHT_EYE_INDICES)
            current_right_state = predict_eye_state(right_eye_img, blink_model)

            if right_eye_state == "Close" and current_right_state == "Open":
                right_blink_count += 1
            right_eye_state = current_right_state

cap.release()

print("FINAL BLINK COUNTS:")
print(f"Right Eye Blinks: {right_blink_count}")
print(f"Left Eye Blinks: {left_blink_count}")

Processing video
FINAL BLINK COUNTS:
Right Eye Blinks: 0
Left Eye Blinks: 37
